<a href="https://colab.research.google.com/github/zackives/upenn-cis5450-hw/blob/main/21_Module_4_Part_III_Real_Time_Streams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Low-Level Streams


## Basic Setup

### Autograder setup

In [ ]:
#PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO
#TO ASSIGN POINTS TO YOU IN OUR BACKEND
STUDENT_ID = 99999999 # YOUR PENN-ID GOES HERE AS AN INTEGER##PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO

In [ ]:
%%writefile notebook-config.yaml

grader_api_url: 'https://23whrwph9h.execute-api.us-east-1.amazonaws.com/default/Grader23'
grader_api_key: 'flfkE736fA6Z8GxMDJe2q8Kfk8UDqjsG3GVqOFOa'

In [ ]:
%set_env HW_ID=cis5450_25f_HW9

In [ ]:
!pip3 install penngrader-client

In [ ]:
import os
from penngrader.grader import *

grader = PennGrader('notebook-config.yaml', os.environ['HW_ID'], STUDENT_ID, STUDENT_ID)

## Installation of Storm, StreamParse and Their Support Libraries

In [ ]:
%set_env STORM_VERSION=1.2.3

In [ ]:
!wget https://raw.githubusercontent.com/technomancy/leiningen/stable/bin/lein

In [ ]:
!wget -nc https://archive.apache.org/dist/storm/apache-storm-$STORM_VERSION/apache-storm-$STORM_VERSION.tar.gz

In [ ]:
!tar zxvf apache-storm-$STORM_VERSION.tar.gz

In [ ]:
!chmod a+x ./lein
!chmod a+x /content/apache-storm-$STORM_VERSION/bin/storm

In [ ]:
%env PATH /opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/apache-storm-1.2.3/bin:/content


In [ ]:
!lein version

In [ ]:
!cp apache-storm-1.2.3/bin/storm apache-storm-1.2.3/bin/storm.bak
!cat apache-storm-1.2.3/bin/storm.bak | sed 's/bin\/python2.6/local\/bin\/python/' > apache-storm-1.2.3/bin/storm

In [ ]:
!storm version

In [ ]:
!pip install streamparse

## Initial Configuration and Execution

Here you can see Streamparse in action, using a sample word count topology.

In [ ]:
import os

os.environ['LEIN_ROOT']='1'

In [ ]:
!sparse quickstart wordcount

We need to make a quick fix to tell StreamParse to use Storm 1.2.3 instead of 1.1.0...

In [ ]:
!cat wordcount/project.clj | sed s/1.1.0/$STORM_VERSION/ > wordcount/project2.clj

In [ ]:
!cat wordcount/project2.clj

In [ ]:
%%writefile project.cl_sec

(require 'cemerick.pomegranate.aether)
(cemerick.pomegranate.aether/register-wagon-factory!
 "http" #(org.apache.maven.wagon.providers.http.HttpWagon.))

In [ ]:
!cat wordcount/project2.clj project.cl_sec > wordcount/project.clj

## Running StreamParse for Real

Run this code and see the output...  At some point you'll want to interrupt it.  Press [Ctrl]-[M][I] when you are done.

In [ ]:
!cat /content/wordcount/project.clj

In [ ]:
!cd /content/wordcount ; sparse run

# Streaming, Joins, and Learning

Let's take a look at how we might use StreamParse spouts and bolts to implement a stream-based incremental machine learning application, for our airline flight data from the High-Level Streaming application.

Here we'll use a combination of Pandas (and PandaSQL) for our incremental joins and grouping, and Creme for incremental machine learning!

For the rest of this code, rather than running directly in a StreamParse topology we'll run operator-by-operator to demonstrate the functionality.

In [ ]:
!pip install duckdb
!pip install river

Let's get our data...

In [ ]:
!wget -nc https://storage.googleapis.com/penn-cis5450/airlines.csv
!wget -nc https://storage.googleapis.com/penn-cis5450/airports.csv
!wget -nc https://storage.googleapis.com/penn-cis5450/ontime.csv
#!wget -nc https://storage.googleapis.com/penn-cis5450/2015-ontime.csv

## Our Spout

Let's define a spout.  To actually run this in StreamParse we would need to put it in a separate source file, which is inconvenient on Colab.

So instead we'll simulate StreamParse and watch what happens.

This spout reads one row at a time from the `ontime` dataframe.

In [ ]:
!mkdir spouts

In [ ]:
%%writefile spouts/flight_spout.py
from streamparse.spout import Spout
import pandas as pd
import duckdb

class FlightSpout(Spout):
  outputs=['YEAR','MONTH','DAY_OF_MONTH','AIRLINE_ID','CARRIER','FL_NUM','ORIGIN','DEST','ARR_DELAY','CANCELLED']
  # Position of next tuple in dataframe
  inx = 0
  # Overall stream tuple id
  tid = 0

  env = False

  # Open the file as a dataframe
  def initialize(self, stormconf, context):
    self.ontime_df = pd.read_csv('/content/ontime.csv')
    self.ontime_df.dropna(inplace=True,subset=['ARR_DELAY_NEW'])
    print('Loaded {} entries'.format(len(self.ontime_df)))
    self.env = (context is not None)
    self.inx = 0

  # Used for showing an example
  def set_stream(self, stream):
    self.stream = stream

  # Read one row, increment the pointer, return the
  # row as a list
  def next_tuple(self):
    tup = self.ontime_df.iloc[self.inx]
    self.inx = self.inx + 1

    # We'll wrap around if we exceed the size of the table
    if self.inx >= len(self.ontime_df):
      self.inx = 0

    # We have to turn the series into a list of columns.
    # There is also an extra, blank column in the file
    self.emit(tup.tolist()[0:-1], tup_id=self.tid)
    self.tid = self.tid + 1

  # If a tuple was processed properly, do nothing
  def ack(self, tup_id):
    pass

  # If a tuple was processed incorrectly, we'll still
  # do nothing
  def fail(self, tup_id):
    pass

  # This lets us run outside Streamparse
  def emit(self, tuple, tup_id):
    if not self.env:
      self.stream.append(tuple)
    else:
      super().emit(tuple, tup_id)


## A Bolt that Joins

As we get tuples, we'll want to join the flight with latitude and longitude information for origin and destination.  We can do this by preloading the `airports` as a dataframe, and incrementally joining tuples as they arrive.

In [ ]:
!mkdir bolts

In [ ]:
%%writefile bolts/join_bolt.py
from streamparse.bolt import Bolt
from pystorm.component import Tuple
import pandas as pd
import duckdb

class JoinBolt(Bolt):
  outputs=['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST','FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','ARR_DELAY']
  env = False
  inx = 0

  def initialize(self, stormconf, context):
    self.airports_df = pd.read_csv('/content/airports.csv')
    self.env = (context is not None)
    self.inx = 0

  def process(self, tup):
    flights_df = pd.DataFrame([tup],columns=['YEAR','MONTH','DAY_OF_MONTH','AIRLINE_ID','CARRIER','FL_NUM','ORIGIN','DEST','ARR_DELAY','CANCELLED'])
    airports_df = self.airports_df
    incremental_results = \
      duckdb.query('select YEAR,MONTH,DAY_OF_MONTH,CARRIER,FL_NUM, ORIGIN, DEST, org.LATITUDE AS FROM_LAT, '\
            'org.LONGITUDE AS FROM_LONG, '\
            'dst.LATITUDE AS TO_LAT, dst.LONGITUDE AS TO_LONG, ARR_DELAY '\
            'from flights_df f join airports_df org on f.origin=org.IATA_CODE '\
            'join airports_df dst on f.dest=dst.IATA_CODE').to_df()
    self.emit(incremental_results.iloc[0].tolist(), self.inx)
    self.inx += 1

  # Used for showing an example
  def set_stream(self, stream):
    self.stream = stream


  # This lets us run outside Streamparse
  def emit(self, tuple, tup_id):
    if not self.env:
      self.stream.append(tuple)
    else:
      super().emit(tuple, tup_id)


## Windowed Grouping

Let's group by day.  We are going to assume that the stream is well-behaved in that days are monotonically increasing in the stream.

Thus, we buffer up tuples as long as they have the same day.  Once the day switches, we group them by the day and compute our statistics, then emit that.

In [ ]:
%%writefile bolts/group_bolt.py
from streamparse.bolt import Bolt
from pystorm.component import Tuple
import pandas as pd
import duckdb

class GroupBolt(Bolt):
  outputs = ['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST',\
                             'FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','NBR_FLIGHTS','AVG_DELAY']
  env = False
  inx = 0

  def initialize(self, stormconf, context):
    self.groups = pd.DataFrame([],
                               columns=['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST','FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','NBR_FLIGHTS','ARR_DELAY'])
    self.last_year = None
    self.last_month = None
    self.last_day_of_month = None
    self.env = (context is not None)
    self.inx = 0

  # Used for showing an example
  def set_stream(self, stream):
    self.stream = stream


  def process(self, tup):
    tup2 = tup[0:]
    if len(tup2) < 13:
      tup2.append(tup[-1])
      tup2[len(tup2)-2] = 1
    # print ("Input: {}".format(len(tup2)))
    flights_df = pd.DataFrame([tup2],
                              columns=['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST','FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','NBR_FLIGHTS','ARR_DELAY'])

    # print ('Flights {}'.format(len(flights_df.columns)))
    if flights_df.iloc[0,0] != self.last_year or flights_df.iloc[0,1] != self.last_month \
      or flights_df.iloc[0,2] != self.last_day_of_month:
      print('** DONE WITH {}-{}-{} with {} entries'.format(self.last_year,self.last_month,self.last_day_of_month,len(self.groups)))
      if len(self.groups) > 0:
        groups_df = self.groups
        grouped_results = duckdb.query('''
                        select YEAR,MONTH,DAY_OF_MONTH,CARRIER,FL_NUM,ORIGIN,DEST,
                        FROM_LAT,FROM_LONG,TO_LAT,TO_LONG, COUNT(ARR_DELAY) as NBR_FLIGHTS, avg(ARR_DELAY) as ARR_DELAY
                        FROM groups_df GROUP BY YEAR,MONTH,DAY_OF_MONTH,CARRIER,FL_NUM,ORIGIN,DEST,FROM_LAT,FROM_LONG,TO_LAT,TO_LONG
                        ORDER BY YEAR,MONTH,DAY_OF_MONTH,CARRIER,FL_NUM,ORIGIN,DEST''').to_df()

        for result in grouped_results.itertuples(index=False):
          self.emit(list(result), self.inx)
          self.inx += 1

      self.groups = flights_df
      self.last_year = flights_df.iloc[0,0]
      self.last_month = flights_df.iloc[0,1]
      self.last_day_of_month = flights_df.iloc[0,2]
    else:
      # print ('Groups {}'.format(len(self.groups.columns)))
      self.groups = pd.concat([self.groups, flights_df])

  # This lets us run outside Streamparse
  def emit(self, tuple, tup_id):
    if not self.env:
      self.stream.append(tuple)
    else:
      super().emit(tuple, tup_id)


## Incremental Learning via Creme (now part of River)

We want to incrementally train a linear regression algorithm to predict our delays.

The Creme / River package (https://github.com/online-ml/river) allows us to do incrementally process data, using stochastic gradient descent-style methods to incrementally perform linear regression.  It also has incremental methods for one-hot encoding, scaling, etc.

In [ ]:
%%writefile bolts/learn_bolt.py
from streamparse.bolt import Bolt
from pystorm.component import Tuple
import pandas as pd
import river.preprocessing
import river.linear_model

class LearnBolt:#(Bolt):
  outputs = ['predicted','actual','error']
  env = False
  inx = 0

  def initialize(self, stormconf, context):
    self.scaler = river.preprocessing.StandardScaler()
    self.lin_reg = river.linear_model.LinearRegression()
    self.carrier_one_hot = river.preprocessing.OneHotEncoder('CARRIER')
    self.origin_one_hot = river.preprocessing.OneHotEncoder('ORIGIN')
    self.dest_one_hot = river.preprocessing.OneHotEncoder('DEST')
    self.env = (context is not None)

  # Used for showing an example
  def set_stream(self, stream):
    self.stream = stream


  def process(self, tup):
    x = pd.Series(tup, index=['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST',\
                             'FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','NBR_FLIGHTS','AVG_DELAY'])

    y = x['AVG_DELAY']
    x = x.drop('AVG_DELAY')
    oh1 = self.carrier_one_hot
    oh1.learn_one(x.to_dict())
    x_onehot = oh1.transform_one(x.to_dict())
    x = x.drop('CARRIER')
    oh2 = self.origin_one_hot
    oh2.learn_one(x)
    x_onehot_1 = oh2.transform_one(x)
    x = x.drop('ORIGIN')
    oh3 = self.dest_one_hot
    oh3.learn_one(x)
    x_onehot_2 = oh3.transform_one(x)
    x = x.drop('DEST')

    # x = x.drop(['CARRIER','AVG_DELAY','ORIGIN','DEST'])
    x = pd.concat([x, pd.Series(x_onehot_1), pd.Series(x_onehot_2)])

    oh4 = self.scaler
    oh4.learn_one(x)
    x = oh4.transform_one(x)
    yhat = self.lin_reg.predict_one(x)
    print('Predicted {}, actual {}, error {}'.format(yhat, y, abs(yhat - y)))
    self.emit([yhat,y,abs(yhat-y)], self.inx)

    self.lin_reg.learn_one(x, y)
    self.inx += 1


  # This lets us run outside Streamparse
  def emit(self, tuple, tup_id):
    if not self.env:
      self.stream.append(tuple)
    else:
      super().emit(tuple, tup_id)


In [ ]:
from spouts.flight_spout import FlightSpout
from bolts.join_bolt import JoinBolt
from bolts.group_bolt import GroupBolt
from bolts.learn_bolt import LearnBolt
import pandas as pd

stream = []
fs = FlightSpout()

fs.initialize(None, None)
fs.set_stream(stream)
for i in range(2000):
  fs.next_tuple()

pd.DataFrame(stream,columns=['YEAR','MONTH','DAY_OF_MONTH','AIRLINE_ID','CARRIER','FL_NUM','ORIGIN','DEST','ARR_DELAY','CANCELLED'])

In [ ]:
stream2 = []
jb = JoinBolt()

jb.initialize(None, None)
jb.set_stream(stream2)

for e in stream:
  jb.process(e)

pd.DataFrame(stream2,columns=['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST','FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','ARR_DELAY'])

In [ ]:
stream3 = []
gb = GroupBolt()

gb.initialize(None, None)
gb.set_stream(stream3)

for e in stream2:
  gb.process(e)

pd.DataFrame(stream3,columns=['YEAR','MONTH','DAY_OF_MONTH','CARRIER','FL_NUM','ORIGIN','DEST',\
                             'FROM_LAT','FROM_LONG','TO_LAT','TO_LONG','NBR_FLIGHTS','AVG_DELAY'])

In [ ]:
stream4 = []
lb = LearnBolt()

lb.initialize(None, None)
lb.set_stream(stream4)

for e in stream3:
  lb.process(e)

pd.DataFrame(stream4,columns=['predicted','actual','error'])

So far we've looked at the bolts separately, just to see what was on the streams.  Let's see them all together in a topology!

In [ ]:
!mkdir topologies

In [ ]:
%%writefile topologies/flight_topology.py
from streamparse import Grouping, Topology


class ModelFlights(Topology):
  flight_spout = FlightSpout.spec()
  join_bolt = JoinBolt.spec(inputs={flight_spout: Grouping.fields("YEAR","MONTH","DAY_OF_MONTH")}, par=2)
  group_bolt = GroupBolt.spec(inputs={join_bolt: Grouping.fields("YEAR","MONTH","DAY_OF_MONTH")}, par=2)
  learn_bolt = LearnBolt.spec(inputs={join_bolt: Grouping.fields("CARRIER","FL_NUM","ORIGIN","DEST")}, par=2)

In [ ]:
%%writefile config.json
{
  "serializer": "json",
  "topology_specs": "topologies/",
  "virtualenv_specs": "virtualenvs/",
  "envs": {
      "prod": {
          "user": "",
          "ssh_password": "",
          "nimbus": "",
          "workers": [],
          "log": {
              "path": "",
              "max_bytes": 1000000,
              "backup_count": 10,
              "level": "info"
          },
          "virtualenv_root": ""
      }
  }
}

## Exercise

Write a *filter bolt* that takes a set of input tuples with schema:

```
['YEAR','MONTH','DAY_OF_MONTH','AIRLINE_ID','CARRIER','FL_NUM','ORIGIN','DEST','ARR_DELAY','CANCELLED']
```

and only emits tuples that arrive in `PHL`.

In [ ]:
%%writefile bolts/filter_bolt.py
from streamparse.bolt import Bolt
from pystorm.component import Tuple
import pandas as pd
import duckdb

class FilterBolt(Bolt):
  outputs=['YEAR','MONTH','DAY_OF_MONTH','AIRLINE_ID','CARRIER','FL_NUM','ORIGIN','DEST','ARR_DELAY','CANCELLED']
  env = False
  inx = 0

  def initialize(self, stormconf, context):
    self.env = (context is not None)
    self.inx = 0

  def process(self, tup):
    # TODO: filter and then emit!

    self.inx += 1

  # Used for showing an example
  def set_stream(self, stream):
    self.stream = stream


  # This lets us run outside Streamparse
  def emit(self, tuple, tup_id):
    if not self.env:
      self.stream.append(tuple)
    else:
      super().emit(tuple, tup_id)


Here's some test code for your bolt.

In [ ]:
from bolts.filter_bolt import FilterBolt

stream5 = []
fb = FilterBolt()

fb.initialize(None, None)
fb.set_stream(stream5)

for e in stream:
  fb.process(e)

pd.DataFrame(stream5,columns=fb.outputs)

In [ ]:
from dill.source import getsource

grader.grade('filter_bolt', getsource(FilterBolt.process))